# 🦕 DINOv2 Optuna + Final Training
### ИСПРАВЛЕНО: validate() возвращает 3 значения

In [22]:
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import optuna
import pandas as pd
import warnings
import gc
import time
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = os.getcwd()
RESULTS_CSV = os.path.join(NOTEBOOK_DIR, 'dinov2_optuna_results.csv')

print(f"Working directory: {NOTEBOOK_DIR}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: e:\marbled_beef
PyTorch: 2.7.1+cu118
CUDA: True
GPU: NVIDIA GeForce RTX 4070


In [23]:
# ====================== CONFIG ======================
class Config:
    seed = 42
    data_dir = "steak"
    model_name = "facebook/dinov2-base"
    hidden_size = 1536
    img_size = 224
    batch_size = 8
    num_workers = 0
    optuna_n_trials = 10
    min_epochs = 15
    max_epochs = 25
    final_epochs = 50
    final_patience = 10
    dropout_rate = 0.4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()

# Seed
import random
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(cfg.seed)

In [24]:
# ====================== DATASET ======================
class SteakDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        self.classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        for cls in self.classes:
            cls_path = os.path.join(root, cls)
            for f in sorted(os.listdir(cls_path)):
                if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                    self.samples.append((os.path.join(cls_path, f), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [25]:
# ====================== TRANSFORMS ======================
train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(45),
    transforms.RandomAffine(degrees=0, scale=(0.8, 1.2), shear=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [26]:
# ====================== LOSS ======================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss

In [27]:
# ====================== MODEL LOADER ======================
def load_dinov2_model(num_classes, device):
    print(f"Loading DINOv2...")
    processor = AutoImageProcessor.from_pretrained(cfg.model_name)
    model = AutoModelForImageClassification.from_pretrained(
        cfg.model_name, num_labels=num_classes, ignore_mismatched_sizes=True
    )
    model.classifier = nn.Sequential(
        nn.Dropout(cfg.dropout_rate),
        nn.Linear(cfg.hidden_size, num_classes)
    )
    model = model.to(device)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"✅ DINOv2 loaded ({num_params/1e6:.1f}M params)")
    return model, processor

In [28]:
# ====================== TRAIN EPOCH ======================
def train_epoch(model, loader, criterion, optimizer, scheduler, epoch, total_epochs, device, processor=None):
    model.train()
    total_loss = correct = total = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{total_epochs}', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        if processor is not None:
            outputs = model(pixel_values=x, labels=y)
            loss = outputs.loss
            logits = outputs.logits
        else:
            logits = model(x)
            loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    scheduler.step()
    return total_loss / max(total, 1), 100. * correct / max(total, 1)

In [29]:
# ====================== VALIDATE ======================
@torch.no_grad()
def validate(model, loader, device, processor=None):
    """Returns: acc, preds, labels (3 values)"""
    model.eval()
    correct = total = 0
    preds, labels = [], []
    pbar = tqdm(loader, desc='Validation', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        if processor is not None:
            outputs = model(pixel_values=x, labels=y)
            logits = outputs.logits
        else:
            logits = model(x)
        pred = logits.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
        preds.extend(pred.cpu().numpy())
        labels.extend(y.cpu().numpy())
    acc = 100. * correct / max(total, 1)
    return acc, preds, labels  # 3 values!

In [30]:
# ====================== FINAL TRAINING ======================
def train_final_model(best_params, train_loader, val_loader, test_loader, num_classes, classes, device):
    print("\n" + "="*60)
    print("FINAL MODEL TRAINING")
    print("="*60)
    
    model, processor = load_dinov2_model(num_classes, device)
    
    if best_params.get('use_focal_loss', True):
        criterion = FocalLoss(
            alpha=best_params.get('focal_alpha', 0.5),
            gamma=best_params.get('focal_gamma', 2.0)
        ).to(device)
    else:
        criterion = nn.CrossEntropyLoss().to(device)
    
    optimizer = optim.AdamW(
        model.parameters(),
        lr=best_params.get('lr', 2e-5),
        weight_decay=best_params.get('weight_decay', 0.05)
    )
    
    scheduler_type = best_params.get('scheduler', 'onecycle')
    if scheduler_type == 'onecycle':
        scheduler = OneCycleLR(
            optimizer, max_lr=best_params.get('lr', 2e-5),
            steps_per_epoch=len(train_loader), epochs=cfg.final_epochs,
            pct_start=0.3, anneal_strategy='cos'
        )
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=cfg.final_epochs, eta_min=1e-7)
    
    best_val_acc = 0
    patience_counter = 0
    history = {'train_acc': [], 'val_acc': []}
    
    for epoch in range(cfg.final_epochs):
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            epoch, cfg.final_epochs, device, processor
        )
        
        # FIX: validate returns 3 values, not 4!
        val_acc, _, _ = validate(model, val_loader, device, processor)
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1:02d}/{cfg.final_epochs:02d} | Train: {train_acc:.2f}% | Val: {val_acc:.2f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), "best_dinov2_model.pth")
            print(f"  → New best! (Val: {val_acc:.2f}%)")
        else:
            patience_counter += 1
        
        if patience_counter >= cfg.final_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(torch.load("best_dinov2_model.pth", map_location=device))
    
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)
    
    test_acc, test_preds, test_labels = validate(model, test_loader, device, processor)
    
    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'Confusion Matrix (Accuracy: {test_acc:.2f}%)')
    plt.tight_layout()
    plt.savefig('dinov2_confusion_matrix.png', dpi=300)
    plt.close()
    
    print(f"\nClassification Report:")
    print(classification_report(test_labels, test_preds, target_names=classes, digits=4))
    
    balanced_acc = balanced_accuracy_score(test_labels, test_preds) * 100
    print(f"\nBalanced Accuracy: {balanced_acc:.2f}%")
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_name': cfg.model_name,
        'classes': classes,
        'accuracy': test_acc,
        'balanced_accuracy': balanced_acc,
        'best_params': best_params,
    }, "dinov2_steak_model.pth")
    
    print(f"\n{'='*60}")
    print("TRAINING COMPLETED!")
    print(f"{'='*60}")
    print(f"✅ Model saved as 'dinov2_steak_model.pth'")
    print(f"🎯 Test Accuracy: {test_acc:.2f}%")
    print(f"⚖️ Balanced Accuracy: {balanced_acc:.2f}%")
    
    return model, test_acc, balanced_acc, history

In [31]:
# ====================== OPTUNA ======================
def objective(trial):
    trial_number = trial.number + 1
    print(f"\n{'='*60}")
    print(f"OPTUNA TRIAL {trial_number}/{cfg.optuna_n_trials}")
    print(f"{'='*60}")
    
    trial_params = {
        'lr': trial.suggest_float('lr', 1e-5, 5e-5, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 0.01, 0.1, log=True),
        'optimizer': trial.suggest_categorical('optimizer', ['adamw']),
        'scheduler': trial.suggest_categorical('scheduler', ['onecycle', 'cosine']),
        'use_focal_loss': trial.suggest_categorical('use_focal_loss', [True, False]),
    }
    
    if trial_params['use_focal_loss']:
        trial_params['focal_alpha'] = trial.suggest_float('focal_alpha', 0.3, 0.7)
        trial_params['focal_gamma'] = trial.suggest_float('focal_gamma', 1.5, 2.5)
    
    num_epochs = 15 if trial_number <= 3 else cfg.max_epochs
    
    start_time = time.time()
    val_acc = train_optuna_trial(
        trial_params, train_loader, val_loader,
        len(dataset.classes), cfg.device, num_epochs
    )
    elapsed_time = time.time() - start_time
    print(f"Trial {trial_number} took {elapsed_time/60:.1f}m, Accuracy: {val_acc:.2f}%")
    return val_acc


def train_optuna_trial(trial_params, train_loader, val_loader, num_classes, device, num_epochs):
    try:
        model, processor = load_dinov2_model(num_classes, device)
        
        if trial_params['use_focal_loss']:
            criterion = FocalLoss(
                alpha=trial_params['focal_alpha'],
                gamma=trial_params['focal_gamma']
            ).to(device)
        else:
            criterion = nn.CrossEntropyLoss().to(device)
        
        optimizer = optim.AdamW(
            model.parameters(),
            lr=trial_params['lr'],
            weight_decay=trial_params['weight_decay']
        )
        
        if trial_params['scheduler'] == 'onecycle':
            scheduler = OneCycleLR(
                optimizer, max_lr=trial_params['lr'],
                steps_per_epoch=len(train_loader), epochs=num_epochs,
                pct_start=0.3, anneal_strategy='cos'
            )
        else:
            scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-7)
        
        best_val_acc = 0
        patience_counter = 0
        
        for epoch in range(num_epochs):
            train_loss, train_acc = train_epoch(
                model, train_loader, criterion, optimizer, scheduler,
                epoch, num_epochs, device, processor
            )
            val_acc, _, _ = validate(model, val_loader, device, processor)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= 5:
                break
        
        del model, optimizer, scheduler, criterion
        torch.cuda.empty_cache()
        gc.collect()
        return best_val_acc
        
    except Exception as e:
        print(f"Trial error: {e}")
        torch.cuda.empty_cache()
        gc.collect()
        return 0.0

In [32]:
# ====================== MAIN ======================
def main():
    global dataset, train_loader, val_loader, test_loader
    
    print("="*60)
    print("🦕 DINOv2 OPTUNA + FINAL TRAINING")
    print("="*60)
    print(f"\nDevice: {cfg.device}")
    print(f"Model: {cfg.model_name}")
    print(f"Data: {cfg.data_dir}")
    
    # Dataset
    print("\nLoading dataset...")
    dataset = SteakDataset(cfg.data_dir)
    print(f"Classes: {dataset.classes}")
    print(f"Total images: {len(dataset)}")
    
    class_counts = [0] * len(dataset.classes)
    for _, label in dataset.samples:
        class_counts[label] += 1
    
    print(f"\nClass distribution:")
    for cls, count in zip(dataset.classes, class_counts):
        print(f"  {cls}: {count} ({count/len(dataset)*100:.1f}%)")
    
    # Split
    idx = np.arange(len(dataset))
    labels = [dataset.samples[i][1] for i in idx]
    train_idx, temp_idx = train_test_split(idx, test_size=0.3, stratify=labels, random_state=cfg.seed)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[labels[i] for i in temp_idx], random_state=cfg.seed)
    print(f"\nSplit: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    
    # DataLoaders
    train_ds = Subset(SteakDataset(cfg.data_dir, transform=train_tfms), train_idx)
    val_ds = Subset(SteakDataset(cfg.data_dir, transform=val_tfms), val_idx)
    test_ds = Subset(SteakDataset(cfg.data_dir, transform=val_tfms), test_idx)
    
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=False)
    
    # Check for existing results
    best_params = None
    if os.path.exists(RESULTS_CSV):
        print(f"\n🔍 Found CSV! Loading best params...")
        df = pd.read_csv(RESULTS_CSV)
        if 'value' in df.columns and len(df) > 0:
            best_idx = df['value'].idxmax()
            best_value = df.loc[best_idx, 'value']
            print(f"   Best trial value: {best_value:.2f}%")
            param_cols = [c for c in df.columns if c.startswith('params_')]
            best_params = {}
            for col in param_cols:
                key = col.replace('params_', '')
                best_params[key] = df.loc[best_idx, col]
            print(f"   Loaded {len(best_params)} parameters")
    
    # Run Optuna if no params
    if best_params is None:
        print("\n" + "="*60)
        print("STARTING OPTUNA OPTIMIZATION")
        print("="*60)
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=cfg.seed),
            pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3)
        )
        study.optimize(objective, n_trials=cfg.optuna_n_trials)
        best_params = study.best_params
        df = study.trials_dataframe()
        df.to_csv(RESULTS_CSV, index=False)
        print(f"Results saved to '{RESULTS_CSV}'")
    
    # Print best params
    print("\n" + "="*60)
    print("BEST PARAMETERS")
    print("="*60)
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    
    # FINAL TRAINING
    model, test_acc, balanced_acc, history = train_final_model(
        best_params, train_loader, val_loader, test_loader,
        len(dataset.classes), dataset.classes, cfg.device
    )


if __name__ == "__main__":
    main()

🦕 DINOv2 OPTUNA + FINAL TRAINING

Device: cuda
Model: facebook/dinov2-base
Data: steak

Loading dataset...
Classes: ['file', 'rib', 'strip']
Total images: 358

Class distribution:
  file: 92 (25.7%)
  rib: 141 (39.4%)
  strip: 125 (34.9%)

Split: Train=250, Val=54, Test=54

🔍 Found CSV! Loading best params...
   Best trial value: 96.30%
   Loaded 7 parameters

BEST PARAMETERS
  focal_alpha: nan
  focal_gamma: nan
  lr: 1.217029388373878e-05
  optimizer: adamw
  scheduler: cosine
  use_focal_loss: False
  weight_decay: 0.0312735303678037

FINAL MODEL TRAINING
Loading DINOv2...


Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1113.20it/s, Materializing param=dinov2.layernorm.weight]                                
Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DINOv2 loaded (86.6M params)


Epoch 01/50 | Train: 46.80% | Val: 70.37%
  → New best! (Val: 70.37%)


Epoch 02/50 | Train: 71.20% | Val: 87.04%
  → New best! (Val: 87.04%)


...

Epoch 26/50 | Train: 99.20% | Val: 94.44%


Epoch 27/50 | Train: 98.40% | Val: 92.59%


Epoch 28/50 | Train: 99.60% | Val: 96.30%


Epoch 29/50 | Train: 99.20% | Val: 96.30%
Early stopping at epoch 29

FINAL EVALUATION



Classification Report:
              precision    recall  f1-score   support

        file     1.0000    0.8571    0.9231        14
         rib     0.8400    1.0000    0.9130        21
       strip     0.8824    0.7895    0.8333        19

    accuracy                         0.8889        54
   macro avg     0.9075    0.8822    0.8898        54
weighted avg     0.8964    0.8889    0.8876        54


Balanced Accuracy: 88.22%

TRAINING COMPLETED!
✅ Model saved as 'dinov2_steak_model.pth'
🎯 Test Accuracy: 88.89%
⚖️ Balanced Accuracy: 88.22%
